# Python 7: errors, logging and testing

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-07-errors-logging-and-testing.ipynb)

A pipeline that fails silently is worse than one that fails loudly. This chapter covers the three habits that make Python code trustworthy in production. First, raise specific errors, and handle only the errors you can really handle. Second, log what happened in a form people can search. Third, test transformations with small, fast checks that run on every change.

Every rule is stated before the code that shows it, and each code cell's `assert` lines are the expected results. If an `assert` fails, your model of that rule is wrong.

## Exceptions: the hierarchy

An **exception** is an object that describes an error. Raising it stops the current block and unwinds the call stack until some `except` handles it. Exceptions form a class hierarchy. `except SomeError` catches that class and every subclass of it.

Two base classes matter. `Exception` is the parent of the errors your code should handle. `BaseException` is the parent of the exceptions that should usually stop the program, such as `KeyboardInterrupt`, which is Ctrl-C. A plain `except Exception` does not catch Ctrl-C, but a bare `except:` does, which is why a bare `except:` is almost always wrong.

In [ ]:
assert issubclass(ValueError, Exception)
assert issubclass(KeyboardInterrupt, BaseException)
assert not issubclass(KeyboardInterrupt, Exception)      # except Exception lets Ctrl-C through

## `try`, `except`, `else` and `finally`

- `try` holds the code that may fail.
- `except` handles one class of exception. Catch the narrowest class that you expect.
- `else` runs only when the `try` block raised nothing.
- `finally` runs **every time**, including after a `return` or a raised error. Use it for cleanup that must happen.

A `return` inside `try` skips `else`, so put the return in `else` when you need `else` to run first.

In [ ]:
trace = []

def read_count(text):
    try:
        value = int(text)
    except ValueError:
        trace.append("except")
        return None
    else:
        trace.append("else")
        return value
    finally:
        trace.append("finally")

assert read_count("5") == 5
assert trace == ["else", "finally"]          # the else ran, then finally ran after the return
trace.clear()
assert read_count("x") is None
assert trace == ["except", "finally"]        # finally runs even when the except returns

### Where you'll use it

- **Transient network errors:** catch `TimeoutError` and retry. Let every other error stop the job, so a real bug is never hidden.
- **Parsing with a fallback:** catch the one error that means "bad text", and count the row as quarantined.

In [ ]:
def parse_amount(text):
    try:
        return float(text)
    except ValueError:                       # expected: the source sent bad text
        return None

assert parse_amount("12.5") == 12.5
assert parse_amount("n/a") is None

## Raising and chaining errors

Raise a specific error with a clear message. Define your own exception classes for the failures your pipeline cares about, and give them a common base class. Callers can then catch `PipelineError` for everything, or a subclass for one failure.

When you convert a low-level error into a higher-level one, use `raise ... from error`. The original error is kept as `__cause__`, and the traceback shows both. Use `raise ... from None` only when the original error is noise.

### Where you'll use it

- **Schema contracts:** a `SchemaError` raised by the loader tells the on-call engineer exactly which columns drifted.
- **Adding context:** a driver error is wrapped with the file name and row number, while the original traceback is kept.

In [ ]:
class PipelineError(Exception):
    """Base class for every error this pipeline raises on purpose."""

class SchemaError(PipelineError):
    """A record does not match the expected columns."""

def require_columns(record, required):
    missing = [column for column in required if column not in record]
    if missing:
        raise SchemaError(f"missing columns: {', '.join(missing)}")
    return record

try:
    require_columns({"id": 1}, ["id", "amount"])
except PipelineError as error:              # the base class catches the subclass
    message = str(error)
    caught = type(error).__name__

assert message == "missing columns: amount"
assert caught == "SchemaError"

def load_amount(record):
    try:
        return float(record["amount"])
    except (KeyError, ValueError) as error:
        raise SchemaError("bad amount") from error

try:
    load_amount({"amount": "x"})
except SchemaError as error:
    assert isinstance(error.__cause__, ValueError)     # the original cause is kept

try:
    try:
        {}["key"]
    except KeyError:
        raise LookupError("no such field") from None
except LookupError as error:
    assert error.__cause__ is None                     # the KeyError is hidden on purpose

## Logging instead of `print`

The `logging` module records messages with a **level**, a time and a **logger name**. Levels let you filter: `DEBUG` for detail, `INFO` for progress, `WARNING` for something odd, and `ERROR` for a failed step. A logger passes on only the messages at or above its level.

Pass values as arguments with `%s` or `%d` placeholders, not as f-strings. The message is built only when the record is actually emitted, so a `DEBUG` line that is filtered out costs almost nothing. Inside an `except` block, `logger.exception` logs the message and the traceback.

### Where you'll use it

- **Operating pipelines:** logs are how an engineer learns why a 3 a.m. run loaded zero rows. INFO lines with counts answer that question.
- **Alerting:** a monitoring tool counts `ERROR` lines and pages the on-call engineer when they pass a threshold.

In [ ]:
import io
import json
import logging

stream = io.StringIO()
logger = logging.getLogger("orders.load")
logger.handlers.clear()
handler = logging.StreamHandler(stream)
handler.setFormatter(logging.Formatter("%(levelname)s %(name)s: %(message)s"))
logger.addHandler(handler)
logger.setLevel(logging.INFO)

logger.debug("hidden: below the level")
logger.info("loaded %d rows from %s", 120, "orders.csv")
logger.warning("skipped %d bad rows", 3)

try:
    load_amount({"amount": "x"})
except SchemaError:
    logger.exception("could not load a record")

# A structured line: one JSON object after the message, easy to parse in a log search tool.
logger.info("run finished %s", json.dumps({"run_id": "r1", "loaded": 120}, sort_keys=True))

lines = stream.getvalue().splitlines()
assert lines[0] == "INFO orders.load: loaded 120 rows from orders.csv"
assert lines[1] == "WARNING orders.load: skipped 3 bad rows"
assert lines[2].startswith("ERROR orders.load: could not load a record")
assert not any("hidden" in line for line in lines)
assert lines[-1] == 'INFO orders.load: run finished {"loaded": 120, "run_id": "r1"}'

In [ ]:
class Expensive:
    """Counts how often its text is built."""
    calls = 0

    def __str__(self):
        Expensive.calls += 1
        return "expensive"

logger.debug("state: %s", Expensive())    # filtered out at INFO, so str() never runs
assert Expensive.calls == 0

logger.info("state: %s", Expensive())     # emitted, so str() runs once
assert Expensive.calls == 1

## Testing transformations

A transformation is a function from input to output, which makes it easy to test. Test the behaviour you care about: the normal case, the empty input, the edge values and the errors. Keep each test small and independent, so a failure points at one rule.

In a notebook, a test is a function whose name starts with `test_` and whose body is `assert` statements. A test runner such as `pytest` finds every `test_` function in a file and reports each failure with its own message. The plain loop below does the same job for this notebook.

### Where you'll use it

- **CI for data code:** a test suite runs on every pull request, so a change to a parsing rule breaks the build instead of production.
- **Regression tests:** when a bad record reaches production, add it as a test case, so the bug cannot come back unnoticed.

In [ ]:
def normalise_country(code):
    if not code or not code.strip():
        raise ValueError("country code is empty")
    return code.strip().upper()

# Table-driven: one list of (input, expected) pairs, one test function.
COUNTRY_CASES = [
    (" de ", "DE"),
    ("nl", "NL"),
    ("Us", "US"),
    ("\tfr\n", "FR"),              # regression case: tabs and newlines from a source file
]

def test_normalise_country_strips_and_uppercases():
    for raw, expected in COUNTRY_CASES:
        assert normalise_country(raw) == expected, f"input {raw!r}"

def test_normalise_country_rejects_blank():
    try:
        normalise_country("   ")
    except ValueError:
        return
    raise AssertionError("expected ValueError")

def test_normalise_country_is_idempotent():
    once = normalise_country("nl")
    assert normalise_country(once) == once

tests = [obj for name, obj in list(globals().items()) if name.startswith("test_") and callable(obj)]
assert len(tests) == 3
for test in tests:
    test()

# An assertion can carry a message. The message appears in the failure report.
try:
    assert normalise_country("de") == "de", "codes are stored in uppercase"
except AssertionError as error:
    assert str(error) == "codes are stored in uppercase"

## Measuring time

`time.perf_counter()` is the clock to use for durations. It is monotonic, so it never goes backwards, and its resolution is high. `time.time()` reads the wall clock, which can jump when the system clock is adjusted. A context manager makes timing reusable.

### Where you'll use it

- **Finding slow steps:** timing each stage of a job shows which one to optimise first.
- **Benchmarks:** comparing two parsing approaches on a sample file before you choose one.

In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def timed(label, log):
    start = time.perf_counter()
    try:
        yield
    finally:
        log.append((label, time.perf_counter() - start))

timings = []
with timed("sum", timings):
    sum(range(1_000_000))
assert timings[0][0] == "sum" and timings[0][1] >= 0

## Practice questions

1. Write `safe_int(text, default)` that returns the integer, or `default` when the text is not an integer. Catch only the errors that `int` raises for bad input.
2. Write `call_with_retries(fn, attempts, state)`. The function `flaky(state)` raises `ConnectionError` on its first two calls and returns `"ok"` after that. Call it up to `attempts` times. Return the first result, or re-raise the last error.
3. Write a test function `test_safe_int_returns_default_for_decimal_text` that checks `safe_int("12.5", 0)` returns `0`, and run it.

## Answers

### Question 1

In [ ]:
def safe_int(text, default):
    try:
        return int(text)
    except (TypeError, ValueError):
        return default

assert safe_int("42", 0) == 42
assert safe_int("x", -1) == -1
assert safe_int(None, 0) == 0

### Question 2

In [ ]:
def call_with_retries(fn, attempts, state):
    last_error = None
    for _ in range(attempts):
        try:
            return fn(state)
        except ConnectionError as error:
            last_error = error
    raise last_error

def flaky(state):
    state["calls"] += 1
    if state["calls"] <= 2:
        raise ConnectionError("temporary")
    return "ok"

state = {"calls": 0}
assert call_with_retries(flaky, 3, state) == "ok"
assert state["calls"] == 3

### Question 3

In [ ]:
def test_safe_int_returns_default_for_decimal_text():
    assert safe_int("12.5", 0) == 0

test_safe_int_returns_default_for_decimal_text()

## Interview notes

- **Catch narrowly and raise specifically.** Every `except Exception: pass` is a bug waiting to hide data loss.
- **Keep the cause.** `raise ... from error` shows both errors in the traceback, which speeds up debugging.
- **`finally` runs after `return`.** Use it for cleanup, and `else` for code that should run only when nothing failed.
- **Log with `%s` placeholders and levels.** Logs are data: searchable, filterable and the first thing read after an incident.
- **Test behaviour, not implementation.** A good test still passes after a refactor that keeps the behaviour the same.
- **Make every bug a test.** Add the failing input to the test table when you fix it, so the regression stays fixed.